# Document loaders

We will examine how to load and split popular file formats. The purpose of is to insert these documents into a vector database for semantic searches.

In [1]:
# Download the NLTK data
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [6]:
!uv pip install langchain-community[all]
!uv pip install docx2txt

Using Python 3.12.13 environment at: /usr
Checked 1 package in 136ms
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 101ms
Prepared 1 package in 6ms
Installed 1 package in 2ms
 + docx2txt==0.9


In [2]:
# Imports
import json, random
from langchain_community.document_loaders import UnstructuredEPubLoader, UnstructuredExcelLoader, UnstructuredPowerPointLoader
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, JSONLoader, UnstructuredXMLLoader
from langchain_community.document_loaders import UnstructuredEmailLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, RecursiveJsonSplitter


/tmp/ipykernel_5219/2840464246.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredEPubLoader, UnstructuredExcelLoader, UnstructuredPowerPointLoader


# Create a text splitter

The purpose of splitting the text is to enable more precise search results by ensuring that each smaller segment captures relevant context within a larger document. When these text fragrments are used as context in prompts, they will not exceed the context window.

In [3]:
# TODO: Create a text splitter
chunk_size = 500
chunk_overlap = 50

text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)



## Loading and splitting documents

Documents can be loaded and split either as a single or as elements eg pages, individual slides, spreadsheet, etc. Set the `mode` arguemnt to
- `single` - load the entire document as a single object default for most loaders
- `multi`, `elements` - split the document into their respective elements eg pages, slides, etc.

In [5]:
def print_chunk_info(chunks):
   print(f'No of chunks: {len(chunks)}')
   idx = random.randrange(0, len(chunks))
   print(f'Chunk index: {idx}')
   print('Chunk details')
   for k, v in enumerate(chunks[idx]):
      print(f'\t{k} = {v}')

Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 149ms
Prepared 5 packages in 41ms
Installed 5 packages in 6ms
 + colorlog==6.10.1
 + doc2txt==1.0.8
 + fast-langdetect==1.0.1
 + fasttext-predict==0.9.2.4
 + robust-downloader==0.0.2


In [7]:
# TODO: Word doucment
word_loader = Docx2txtLoader(file_path="/content/docs/SST RL Python Setup.docx")

chunks = word_loader.load_and_split(text_splitter)

In [9]:
print(len(chunks))
print_chunk_info(chunks)

8
No of chunks: 8
Chunk index: 2
Chunk details
	0 = ('id', None)
	1 = ('metadata', {'source': '/content/docs/SST RL Python Setup.docx'})
	2 = ('page_content', 'Create a new environment\n\nconda create --name myenv \n\n\n\nwhere myenv is the name of your environment. Feel free to use any name. You can create multiple environment\n\n\n\nNote: You can confirm the newly created environment, by listing all envs\n\nconda env list\n\n\n\nActivate your environment\n\nOnce you have created your environment, you will need to activate it\n\n\tconda activate myenv\n\n\n\nYour command prompt will now include the environment’s name.\n\n\n\nExit from your environment')
	3 = ('type', 'Document')


In [13]:
!uv pip install pypdf

Using Python 3.12.13 environment at: /usr
Resolved 1 package in 107ms
Prepared 1 package in 21ms
Installed 1 package in 3ms
 + pypdf==6.13.1


In [6]:
# TODO: Load PDF
pdf_loader = PyPDFLoader(file_path="/content/docs/Path-to-GitOps-Red-Hat-Developer-e-book.pdf")

chunks = pdf_loader.load_and_split(text_splitter)

print_chunk_info(chunks)


No of chunks: 239
Chunk index: 165
Chunk details
	0 = ('id', None)
	1 = ('metadata', {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2022-07-15T19:22:48-05:00', 'moddate': '2022-07-17T10:32:28-04:00', 'title': 'The Path to GitOps', 'source': '/content/docs/Path-to-GitOps-Red-Hat-Developer-e-book.pdf', 'total_pages': 45, 'page': 31, 'page_label': '32'})
	2 = ('page_content', 'Chapter 5 – Repository and Directory Structures\nThe Path to GitOps | 32\nrepository secrets, and configuration files specific to the Git controller, Argo CD. \nEach configuration has its own directory.\n•   core: This contains Y AML for the core functionality of the cluster. The Kubernetes \nadministrator places resources here that are necessary for the functionality of the \ncluster, such as cluster configurations and cluster workloads.')
	3 = ('type', 'Document')


In [20]:
!uv pip install unstructured
!uv pip install pypandoc

Using Python 3.12.13 environment at: /usr
Checked 1 package in 138ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 130ms


In [7]:
# TODO: Load EPUB
epub_loader = UnstructuredEPubLoader(file_path="/content/docs/The_Chocolate_Box-Agatha_Christie.epub")

chunks = epub_loader.load_and_split(text_splitter)

print_chunk_info(chunks)


[WARNING] Sandbox argument was used, but pandoc version is too low. Ignoring argument.


No of chunks: 80
Chunk index: 55
Chunk details
	0 = ('id', None)
	1 = ('metadata', {'source': '/content/docs/The_Chocolate_Box-Agatha_Christie.epub'})
	2 = ('page_content', '"There is no news, monsieur?"\n\n"None as yet, my friend."\n\n"Ah Pauvre Monsieur Déroulard!" he sighed. "I too was of his way of thinking. I do not care for priests. Not that I would say so in the house. The women are all devout-a good thing perhaps. Madame est très pieuse-et Mademoiselle Virginie aussi.\' Mademoiselle Virginie? Was she "très pieuse"? Thinking of the tear-stained passionate face I had seen that first day, I wondered.')
	3 = ('type', 'Document')


## Processing structured document/JSON

In [9]:
!uv pip install jq

Using Python 3.12.13 environment at: /usr
Resolved 1 package in 148ms
Prepared 1 package in 20ms
Installed 1 package in 1ms
 + jq==1.11.0


In [10]:
# TODO: Extract specific attributes from the JSON document, use JSON path to define which element
json_loader = JSONLoader(
    file_path="/content/docs/tv-shows.json",
    jq_schema='.[].summary',
    text_content=True
)

chunks = json_loader.load_and_split(text_splitter)

print_chunk_info(chunks)


No of chunks: 392
Chunk index: 16
Chunk details
	0 = ('id', None)
	1 = ('metadata', {'source': '/content/docs/tv-shows.json', 'seq_num': 12})
	2 = ('page_content', '<p><b>Lost Girl</b> follows supernatural seductress Bo, a Succubus who feeds on the sexual energy of humans. Growing up with human parents, Bo had no reason to believe she was anything other than the girl next door — until she drained her boyfriend to death in their first sexual encounter. Now she has hit the road alone and afraid. <br> She discovers she is one of the Fae, creatures of legend and folklore, who pass as humans while feeding off them secretly and in different ways, as they have for')
	3 = ('type', 'Document')
